## Workspace setup

This notebook is used to convert root files to TF Datasets for model training.

In [3]:
from datetime import datetime 
import uproot
import awkward as ak
import tensorflow as tf
import numpy as np
import importlib
from functools import partial
from termcolor import colored

from tensorflow.data import Dataset, TFRecordDataset
from tensorflow.data.experimental import TFRecordWriter
from tensorflow.train import BytesList, FloatList, Int64List
from tensorflow.train import Example, Features, Feature
import tensorflow_datasets as tfds

# Notebooks run from VSCode use home directory as a base path
# while notebooks run from JupyterLab use the current directory as a base path
import sys, os

#setting up the working directory and the additional .py files directory
user = os.getenv("USER")
if user == None:
    sys.path.append("/home/jovyan/TPCReco/PythonAnalysis/python/")
    os.chdir("/home/jovyan/TPCReco/PythonAnalysis/")
else:
    sys.path.append("/home/akalinow/scratch/ELITPC/TPCReco/PythonAnalysis/python/")
    os.chdir('/home/akalinow/scratch/ELITPC/PythonAnalysis/')

import io_functions as io
import utility_functions as utils

#creating data directory
from pathlib import Path
Path("data").mkdir(parents=True, exist_ok=True)

In [4]:
#MAKE SURE THIS PATH IS CORRECT 
dataPath = "../resources/SimEvent_Track3D_TwoProng_gun_MC_10k.root"
print("exists:", os.path.exists(dataPath))


#CHECk WHETHER THE DATAFILE IS DAMAGED
with open(dataPath, "rb") as f:
    pass

try:
    f = uproot.open(dataPath)
    print("opened by uproot")
    print("keys:", f.keys())
    print("classnames:", f.classnames())
except Exception as e:
    print("uproot error:", repr(e))

exists: True
opened by uproot
keys: ['TPCData;1']
classnames: {'TPCData;1': 'TTree'}


### Convert simulated data.

Simulated data contains Track3D objects, for generated and reconstructed tracks.
We create TFRecords for both SimEvent and RecoEvent. Both TFRecords should appear in the 'data' directory.

In [5]:
%%time
importlib.reload(io)

rootFiles = [dataPath+':TPCData']

# Convert ROOT files to TF format and save to output directory
simOutputDir = './data/'+dataPath[13:-5]
print('sim output directory: ', simOutputDir)
io.convertROOT(rootFiles, simOutputDir, fields= io.simEventFields)

print()

# uproot always takes the first branch with given name, unless 
# explicit branch is given as input path. Filtering by branches in
# iterate does not work. We have to give full path to the branch: TPCData/RecoEvent
rootFiles = [aFile+'/RecoEvent' for aFile in rootFiles]

# Convert ROOT files to TF format and save to output directory
recoOutputDir = './data/'+'Reco'+dataPath[16:-5]
print('reco output directory: ', recoOutputDir)
io.convertROOT(rootFiles, recoOutputDir, fields=io.recoEventFields)

sim output directory:  ./data/SimEvent_Track3D_TwoProng_gun_MC_10k


I0000 00:00:1787845548.481242     937 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 43475 MB memory:  -> device: 0, name: NVIDIA L40S, pci bus id: 0000:89:00.0, compute capability: 8.9
I0000 00:00:1787845548.482286     937 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 43475 MB memory:  -> device: 1, name: NVIDIA L40S, pci bus id: 0001:c5:00.0, compute capability: 8.9



reco output directory:  ./data/RecoEvent_Track3D_TwoProng_gun_MC_10k
CPU times: user 2min 15s, sys: 15.8 s, total: 2min 31s
Wall time: 1min 52s


### Merge SimEvent and RecoEvent data
Merges the simulated and the reconstructed dataset into one.

In [6]:
simDataset = tf.data.Dataset.load(simOutputDir, compression="GZIP")
recoDataset = tf.data.Dataset.load(recoOutputDir, compression="GZIP")

mergedDataset = tf.data.Dataset.zip((simDataset, recoDataset))
mergedDataset = mergedDataset.map(
    lambda sim, reco: {
        'sim': sim,
        'reco': reco})

# filter merged dataset
# maxAlphaLength = 150.0

# mergedDataset = mergedDataset.filter(
#     lambda x: tf.reduce_all(
#         tf.less_equal(
#             tf.sqrt(
#                 tf.reduce_sum(
#                     tf.square(x["sim"][1][:, 0] - x["sim"][1][:, 1]),
#                     axis=-1
#                 )
#             ),
#             maxAlphaLength
#         )
#     )
# )

# save merged dataset
mergedOutputDir = './data/'+'Merged'+dataPath[16:-5]
print('merged output dir: ', mergedOutputDir)
mergedDataset.save(mergedOutputDir, compression="GZIP")

merged output dir:  ./data/MergedEvent_Track3D_TwoProng_gun_MC_10k


### Load the dataset and create a dataframe in XYZ coordinates

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

raw_dataset = tf.data.Dataset.load(
    mergedOutputDir,
    compression="GZIP"
)

dataset = raw_dataset.map(
    lambda event: (
        event["sim"][0],
        event["sim"][1]
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

dataset = dataset.map(
    lambda x, y: (
        tf.reshape(x, io.projections.shape),
        tf.reshape(y, (9,))
    ),
    num_parallel_calls=tf.data.AUTOTUNE
)

In [8]:
%%time
# ============================================================
# Build dataframe with SIM labels only
# ============================================================
rows = []

for features, labels in dataset:
    rows.append(labels.numpy())

data = np.vstack(rows)

df_XYZ = pd.DataFrame(
    data=data,
    columns=utils.columnsXYZ
)

df_XYZ.describe()

CPU times: user 1min 2s, sys: 255 ms, total: 1min 3s
Wall time: 1min


2026-08-27 16:04:46.909597: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


,xVtx,xAlpha,xCarbon,yVtx,yAlpha,yCarbon,zVtx,zAlpha,zCarbon
count,8829.000000,8829.000000,8829.000000,8829.000000,8829.000000,8829.000000,8829.000000,8829.000000,8829.000000
mean,0.572337,-0.654614,-0.058913,-0.013039,0.344699,-0.068853,-35.741943,-36.611862,-35.667976
std,57.083385,64.082985,57.945415,1.013962,32.717094,5.596666,23.207069,27.912384,27.981234
min,-99.975624,-139.731125,-112.539452,-4.230233,-89.654823,-14.476206,-52.943466,-66.150398,-52.946159
25%,-48.231468,-52.504597,-48.918587,-0.701887,-17.531693,-4.322527,-49.509495,-52.936562,-52.946159
50%,0.684229,-1.541533,-0.131783,-0.005115,0.274331,-0.062518,-45.313549,-52.311214,-52.239059
75%,49.705441,51.117260,49.795982,0.676219,18.513618,4.206935,-33.439419,-30.021870,-28.370626
max,99.963669,139.939758,112.669235,3.339547,89.901985,14.155437,47.855301,66.005203,65.962181


Analyze the contents of the dataframe. Visualize with plots if needed

In [9]:
df_XYZ.head()

,xVtx,xAlpha,xCarbon,yVtx,yAlpha,yCarbon,zVtx,zAlpha,zCarbon
0,36.633728,-65.251015,46.038345,-0.097890,60.112083,-6.120940,38.436047,-52.946159,47.739853
1,71.835655,-5.573812,81.827919,0.388468,2.399107,0.279554,-46.748501,-7.826473,-52.946159
2,-84.533318,-47.088375,-93.975349,-0.175817,-12.519833,2.549979,-40.662319,-52.554863,-37.401409
3,27.613329,50.649536,19.821629,0.173068,11.229851,-3.510020,-48.333412,-52.862751,-47.434006
4,-85.260765,-108.727242,-81.147675,-1.130165,-8.106128,0.596651,-20.874544,-52.946159,-13.193948
